In [4]:
import os
import sys
import datetime
import pandas as pd
import time
import traceback
import _pickle as pickle

# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
upper_dir = '/home/trade_core' # TEST
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger
from exchange_plugin.upbit_plug import UserUpbitAdaptor
from exchange_plugin.binance_plug import UserBinanceAdaptor
from etc.redis_connector.redis_connector import InitRedis

In [5]:
binance_access_key = '4PFfYsObYyUaMlk7m6tT9qIl8X8P3WCUsyu1lEyZ4h50VfTMPIQCNdL9eOJCl3Lb'
binance_secret_key = '011Z1W9p4425AZgtCNbs5L1d77ehZFNmBIwT0pz1SJGP7EiOlPILfWBglbVsxMcK'

upbit_access_key = "x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8"
upbit_secret_key = "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL"

okx_access_key = "fcb66a6e-0443-4743-8d9b-61caf7eab662"
okx_secret_key = "0029E09874B38F5AC7E68E9DFC348667"
okx_passphrase = "121431Rn!!"

bybit_access_key = "S3YKBbD0kz1WwcfqZD"
bybit_secret_key = "390M6dKJ9J0uEK7vXYAl3qxVCh944tRrzHah"

In [9]:
import os
import sys
import datetime
import pandas as pd
import time
import traceback
import _pickle as pickle

upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger
from exchange_plugin.upbit_plug import UserUpbitAdaptor
from exchange_plugin.binance_plug import UserBinanceAdaptor
from etc.redis_connector.redis_connector import InitRedis

class UserExchangeAdaptor:
    def __init__(self, logging_dir=None):
        self.logger = KimpBotLogger("integrated_plug", logging_dir=logging_dir).logger
        self.logger.info('integrated_plug_logger init')
        self.exchange_adaptor_dict = {}
        self.exchange_adaptor_dict["UPBIT"] = UserUpbitAdaptor(logging_dir=logging_dir)
        self.exchange_adaptor_dict["BINANCE"] = UserBinanceAdaptor(logging_dir=logging_dir)
        # redis connection to read info_dict
        self.local_redis_client = InitRedis(host='localhost', port=6379, db=0, passwd=None)

    def check_api_key(self, exchange, access_key, secret_key, passphrase=None, futures=False):
        """FUTURES includes USD_M, COIN_M"""
        exchange = exchange.upper()
        if exchange not in self.exchange_adaptor_dict:
            raise Exception(f'exchange {exchange} not supported')
        if exchange == "OKX":
            return self.exchange_adaptor_dict[exchange].check_api_key(access_key, secret_key, passphrase, futures)
        else:
            return self.exchange_adaptor_dict[exchange].check_api_key(access_key, secret_key, futures)
    
    def get_position(self, exchange, access_key, secret_key, market_type, passphrase=None):
        """SPOT position columns: ['asset', 'free', 'locked']
        USD_M position columns: ["symbol", "base_asset", "qty", "margin_type", "entry_price", "liquidation_price", "leverage"]
        """
        exchange = exchange.upper()
        if exchange not in self.exchange_adaptor_dict:
            raise Exception(f'exchange {exchange} not supported')
        exchange_adaptor = self.exchange_adaptor_dict[exchange]
        if exchange == "UPBIT":
            position_df = exchange_adaptor.get_balance(access_key, secret_key, market_type)
        elif exchange == "BINANCE":
            position_df = exchange_adaptor.all_position_information(access_key, secret_key, market_type)
            info_df = pickle.loads(self.local_redis_client.get_data(f'TRADE_CORE|{exchange.lower()}_{market_type.lower()}_info_df'))
            position_df = position_df.merge(info_df[['symbol','base_asset']], how='left', on='symbol')
            position_df = position_df.rename(columns={"positionAmt":"qty", "marginType":"margin_type", "entryPrice":"entry_price", "liquidationPrice":"liquidation_price"})
            position_df["ROI"] = position_df.apply(lambda x: (x['entry_price']-x['markPrice'])/x['markPrice']*x['leverage']*100 if x['qty']<0 else 
                                                   (x['markPrice']-x['entry_price'])/x['entry_price']*['leverage']*100, axis=1)
        else:
            raise Exception(f'exchange {exchange} not supported')
        return position_df
    
    def get_capital(self, exchange, access_key, secret_key, market_type, passphrase=None):
        exchange = exchange.upper()
        if exchange not in self.exchange_adaptor_dict:
            raise Exception(f'exchange {exchange} not supported')
        exchange_adaptor = self.exchange_adaptor_dict[exchange]
        if exchange == "UPBIT":
            currency = 'KRW'
            position_df = exchange_adaptor.get_balance(access_key, secret_key, market_type)
            ticker_df = pickle.loads(self.local_redis_client.get_data(f"TRADE_CORE|{exchange.lower()}_{market_type.lower()}_ticker_df"))
            position_df['symbol'] = position_df['unit_currency']+'-'+position_df['asset']
            merged_df = position_df.merge(ticker_df[['symbol','lastPrice']], how='left', on='symbol')
            merged_df.loc[merged_df['asset']==currency, 'lastPrice'] = 1
            merged_df['entered'] = merged_df['avg_buy_price'] * merged_df['free']
            merged_df['locked'] = merged_df['avg_buy_price'] * merged_df['locked']
            free = round(merged_df.loc[merged_df['asset']==currency, 'free'].values[0])
            locked = round(merged_df['entered'].sum() + merged_df['locked'].sum())
            before_pnl = round(free + locked)
            after_pnl = round((((merged_df['free'] + merged_df['locked']) * merged_df['lastPrice'])).sum())
            pnl = round(after_pnl - before_pnl)
        elif exchange == "BINANCE":
            currency = 'USDT'
            balance_df = self.exchange_adaptor_dict[exchange].get_balance(access_key, secret_key, market_type)
            currency_df = balance_df[balance_df['asset']==currency]
            free = round(currency_df['availableBalance'].values[0],1)
            locked = round(currency_df['walletBalance'].values[0] - free,1)
            before_pnl = round(currency_df['walletBalance'].values[0],1)
            pnl = round(currency_df['unrealizedProfit'].values[0],1)
            after_pnl = round(currency_df['walletBalance'].values[0] + pnl,1)
        else:
            raise Exception(f'exchange {exchange} not supported')

        capital_series = pd.Series({'free':free, 'locked':locked, 'before_pnl': before_pnl, 'pnl':pnl, 'after_pnl':after_pnl, 'currency': currency})
        return capital_series

In [10]:
user_exchange_adaptor = UserExchangeAdaptor()

2024-02-20 05:14:27,344 - integrated_plug - INFO - integrated_plug_logger init
2024-02-20 05:14:27,344 - integrated_plug - INFO - integrated_plug_logger init
2024-02-20 05:14:27,346 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-20 05:14:27,346 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-20 05:14:27,347 - user_binance_plug - INFO - user_binance_plug_logger started.
2024-02-20 05:14:27,347 - user_binance_plug - INFO - user_binance_plug_logger started.


In [22]:
user_exchange_adaptor.check_api_key('upbit', upbit_access_key, binance_secret_key, futures=False)

(False,
 '"None of [Index([\'balance\', \'locked\', \'avg_buy_price\'], dtype=\'object\')] are in the [columns]"')

In [3]:
ticker_df = pickle.loads(user_exchange_adaptor.local_redis_client.get_data('TRADE_CORE|binance_usd_m_ticker_df'))
ticker_df

NameError: name 'pickle' is not defined

In [11]:
capital_series = user_exchange_adaptor.get_capital('upbit', upbit_access_key, upbit_secret_key, 'SPOT')
capital_series

free           34201706
locked         74424453
before_pnl    108626159
pnl            39629058
after_pnl     114053511
currency            KRW
dtype: object

In [12]:
capital_series = user_exchange_adaptor.get_capital('binance', binance_access_key, binance_secret_key, 'USD_M')
capital_series.to_json()

'{"free":14094.1,"locked":18535.4,"before_pnl":32629.5,"pnl":-4053.9,"after_pnl":28575.6,"currency":"USDT"}'

In [6]:
client = user_exchange_adaptor.exchange_adaptor_dict['BINANCE'].load_user_client(binance_access_key, binance_secret_key)
client.futures_account()

{'feeTier': 0,
 'canTrade': True,
 'canDeposit': True,
 'canWithdraw': True,
 'tradeGroupId': -1,
 'updateTime': 0,
 'multiAssetsMargin': False,
 'totalInitialMargin': '14412.20905326',
 'totalMaintMargin': '754.91440340',
 'totalWalletBalance': '32629.51116029',
 'totalUnrealizedProfit': '-3781.85109985',
 'totalMarginBalance': '28847.66006044',
 'totalPositionInitialMargin': '14412.20905326',
 'totalOpenOrderInitialMargin': '0.00000000',
 'totalCrossWalletBalance': '32629.51116029',
 'totalCrossUnPnl': '-3781.85109985',
 'availableBalance': '14422.03870033',
 'maxWithdrawAmount': '14422.03870033',
 'assets': [{'asset': 'BTC',
   'walletBalance': '0.00000000',
   'unrealizedProfit': '0.00000000',
   'marginBalance': '0.00000000',
   'maintMargin': '0.00000000',
   'initialMargin': '0.00000000',
   'positionInitialMargin': '0.00000000',
   'openOrderInitialMargin': '0.00000000',
   'maxWithdrawAmount': '0.00000000',
   'crossWalletBalance': '0.00000000',
   'crossUnPnl': '0.00000000',
